# AI工学101 — 第26回

## 実データの前処理：欠損値・カテゴリ変数・ColumnTransformer

第25回までは、かなり重要なところまで来ました。

```text
モデル
↓
確率
↓
Threshold
↓
Precision / Recall
↓
運用
```

今回は、いったんモデルから離れて、**実際のデータを機械学習に食わせられる形にする工程**を扱います。

ここから一気に実務っぽくなります。

これまでのデータは、

```text
全部数値
欠損なし
きれいな配列
```

でした。

でも現実のデータは、例えば、

```text
年齢：35
年収：4500000
職種：エンジニア
地域：京都
年収：欠損
```

みたいになります。

つまり、

* 数値データ
* 文字列データ
* 欠損値

が混在します。

このままでは多くの機械学習モデルに投入できません。

そこで今日は、

> **「汚い表データ → 機械学習可能な特徴量」**

という変換パイプラインを作ります。

---

# 🎯 今日のゴール

今日できるようになること：

* 欠損値を扱える
* 数値特徴量を補完できる
* カテゴリ変数をOne-Hot Encodingできる
* `ColumnTransformer` を使える
* `Pipeline` の中に前処理をまとめられる
* **前処理もtrainデータだけで学習する**というデータリーク対策を理解する

---

# 📖 講義：約20分

## 1. 欠損値とは？

例えば、

| 年齢 |  年収 | 職種       |
| -: | --: | -------- |
| 25 | 300 | Engineer |
| 32 | 450 | Designer |
| 41 | NaN | Engineer |
| 28 | 350 | NaN      |

この `NaN` が欠損値です。

機械学習モデルによっては、

```python
model.fit(X, y)
```

にそのまま渡せません。

そこで、

> **欠損値補完（Imputation）**

を行います。

---

# 📖 2. 欠損値をどう埋める？

代表的なのが、

### 平均値

```text
300
450
NaN
350
```

なら、

```text
平均 = 366.7
```

として埋める。

---

### 中央値

```text
300
350
450
```

なら、

```text
中央値 = 350
```

を使う。

外れ値が多い場合、平均より中央値が頑健なことがあります。

---

# 📖 3. カテゴリ変数

例えば、

```text
職種
Engineer
Designer
Engineer
Manager
```

これは文字列なので、そのままでは多くのモデルに入力できません。

そこで、

**One-Hot Encoding**

を使います。

例えば、

```text
Engineer
Designer
Manager
```

を、

```text
Engineer  Designer  Manager
    1         0         0
    0         1         0
    1         0         0
    0         0         1
```

に変換します。

---

# 💡 なぜ「0,1,2」にしないの？

例えば、

```text
Engineer = 0
Designer = 1
Manager = 2
```

とすると、

モデルによっては、

```text
Manager > Designer > Engineer
```

という**意味のない順序関係**があるように解釈してしまいます。

カテゴリに順序がないなら、

```text
One-Hot Encoding
```

のほうが自然です。

---

# 📖 4. 数値とカテゴリで処理が違う

ここが今日の核心。

```text
年齢
年収
  ↓
数値処理
  ↓
欠損補完
  ↓
StandardScaler
```

一方、

```text
職種
地域
  ↓
カテゴリ処理
  ↓
欠損補完
  ↓
One-Hot Encoding
```

つまり、

**列によって前処理が違います。**

これをまとめるために登場するのが、

> `ColumnTransformer`

です。

---

# 💻 実習1：現実っぽいデータを作る

まずDataFrameを作ります。

```python
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "age": [
        25, 32, 41, 28, 36, 50, 29, 45
    ],
    "income": [
        300, 450, np.nan, 350,
        520, 700, np.nan, 610
    ],
    "job": [
        "Engineer",
        "Designer",
        "Engineer",
        "Manager",
        "Designer",
        "Manager",
        np.nan,
        "Engineer"
    ],
    "city": [
        "Kyoto",
        "Osaka",
        "Kyoto",
        "Tokyo",
        "Osaka",
        "Tokyo",
        "Kyoto",
        np.nan
    ]
})
```

確認。

```python
print(df)
```

---

# 💻 実習2：欠損値を確認

```python
print(
    df.isna().sum()
)
```

例えば、

```text
age       0
income    2
job       1
city      1
```

のようになります。

これで、

> **どの列に欠損があるか**

確認できます。

---

# 💻 実習3：ターゲットを追加

今回は、

```text
採用されたか？
```

を予測することにします。

```python
y = np.array([
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    1
])
```

特徴量からターゲットを分離。

```python
X = df
```

---

# 💻 実習4：train/test分割

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
```

ここで重要。

**欠損値補完をまだやりません。**

先に分割します。

---

# 🧠 なぜ？

例えば全データの平均値を使って、

```python
income.fillna(
    income.mean()
)
```

をやってから分割すると、

**testデータの情報が前処理に混ざります。**

これはデータリークです。

正しくは、

```text
全データ
 ↓
train / test
 ↓
trainだけから補完ルールを学習
 ↓
train/testへ適用
```

です。

Pipelineは、この事故を防ぎやすくしてくれます。

---

# 💻 実習5：列を分ける

```python
numeric_features = [
    "age",
    "income"
]

categorical_features = [
    "job",
    "city"
]
```

---

# 💻 実習6：数値データの前処理

```python
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
```

数値用Pipeline。

```python
numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "scaler",
        StandardScaler()
    )
])
```

流れは、

```text
数値データ
 ↓
中央値で欠損補完
 ↓
StandardScaler
```

です。

---

# 💻 実習7：カテゴリデータの前処理

```python
from sklearn.preprocessing import OneHotEncoder
```

カテゴリ用Pipeline。

```python
categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])
```

流れは、

```text
カテゴリ
 ↓
最頻値で欠損補完
 ↓
One-Hot Encoding
```

です。

---

# 🧠 `handle_unknown="ignore"` が重要

例えばtrainデータには、

```text
Kyoto
Osaka
Tokyo
```

しかなかったのに、

testデータに、

```text
Nagoya
```

が出てきたとします。

その場合、

```python
OneHotEncoder(
    handle_unknown="ignore"
)
```

としておくと、未知カテゴリに遭遇してもエラーになりにくくなります。

実務ではかなり大事な設定です。

---

# 💻 実習8：ColumnTransformer

いよいよ今日の主役。

```python
from sklearn.compose import ColumnTransformer
```

作成。

```python
preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])
```

これで、

```text
age
income
   ↓
数値Pipeline

job
city
   ↓
カテゴリPipeline
```

という分岐を作れました。

---

# 💻 実習9：実際に変換してみる

```python
X_train_transformed = (
    preprocessor.fit_transform(
        X_train
    )
)
```

testは、

```python
X_test_transformed = (
    preprocessor.transform(
        X_test
    )
)
```

です。

ここも超重要。

```text
train → fit_transform()
test  → transform()
```

です。

**testではfitしません。**

---

# 💻 実習10：変換後のサイズを見る

```python
print(
    X_train.shape
)

print(
    X_train_transformed.shape
)
```

元データは、

```text
4列
```

ですが、

One-Hot Encodingによってカテゴリ列が複数列に展開されるので、

**特徴量数が増えます。**

---

# 💻 実習11：最終的にモデルまでつなぐ

ここまでを一つのPipelineにします。

```python
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

これで、

```text
生データ
 ↓
欠損値補完
 ↓
数値標準化
 ↓
カテゴリOne-Hot
 ↓
Logistic Regression
```

が一本につながりました。

---

# 💻 実習12：学習

```python
model.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = model.predict(
    X_test
)
```

評価。

```python
from sklearn.metrics import accuracy_score

print(
    accuracy_score(
        y_test,
        pred
    )
)
```

これで、

**生のDataFrameをそのままPipelineに渡して学習**

できました。

---

# 🧠 今日の重要ポイント

これまで、

```text
NumPy配列
 ↓
モデル
```

だったのが、

今日は、

```text
現実のDataFrame
 ↓
ColumnTransformer
 ↓
欠損値処理
 ↓
カテゴリ変換
 ↓
数値標準化
 ↓
モデル
```

になりました。

これはかなり大きなステップです。

---

# 💻 実習13：Cross Validationまで接続する

第18回まで戻ります。

```python
from sklearn.model_selection import cross_validate
```

評価。

```python
results = cross_validate(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)
```

平均。

```python
print(
    results[
        "test_score"
    ].mean()
)
```

ここでも、

Pipelineのおかげで各Foldについて、

```text
train Fold
 ↓
前処理fit
 ↓
モデルfit

validation Fold
 ↓
前処理transform
 ↓
モデルpredict
```

という正しい処理が行われます。

---

# 📖 5. ColumnTransformerがなぜ重要なのか

今日の内容を概念図にすると、

```text
                    生データ
                       │
              ┌────────┴────────┐
              ↓                 ↓
          数値列              カテゴリ列
              │                 │
          Imputer           Imputer
              │                 │
          Scaler            Encoder
              │                 │
              └────────┬────────┘
                       ↓
                    結合
                       ↓
                     Model
```

です。

つまり、

> **「列ごとに異なる前処理を行い、最後にモデルへ渡す」**

という機械学習パイプラインを構築できます。

---

# ✍️ 演習

## 問1

次のデータについて、

```text
age
income
job
city
```

を、

```text
数値特徴量
カテゴリ特徴量
```

に分類してください。

---

## 問2

`income` の欠損値を中央値で補完する `SimpleImputer` を作ってください。

---

## 問3

`job` と `city` を、

```python
OneHotEncoder(
    handle_unknown="ignore"
)
```

で変換してください。

---

## 問4

`ColumnTransformer` を作り、

```text
数値
→ Imputer
→ StandardScaler

カテゴリ
→ Imputer
→ OneHotEncoder
```

という処理を実装してください。

---

## 問5

これを、

```text
前処理
↓
Logistic Regression
```

のPipelineにまとめてください。

---

## 問6

Cross Validationを使って評価してください。

---

# 👾 ボス戦：実務型Pipeline

次のようなデータを想定します。

```python
df = pd.DataFrame({
    "age": [25, 32, np.nan, 41, 29, 50],
    "income": [300, 450, 380, np.nan, 350, 700],
    "job": [
        "Engineer",
        "Designer",
        "Engineer",
        "Manager",
        np.nan,
        "Manager"
    ],
    "city": [
        "Kyoto",
        "Osaka",
        "Tokyo",
        "Kyoto",
        "Osaka",
        np.nan
    ]
})
```

ターゲット：

```python
y = np.array([
    0,
    0,
    1,
    1,
    0,
    1
])
```

以下を一つのPipelineとして完成させてください。

```text
DataFrame
 ↓
ColumnTransformer
 ├─ 数値
 │   ├─ median imputation
 │   └─ StandardScaler
 │
 └─ カテゴリ
     ├─ most frequent imputation
     └─ One-Hot Encoding
 ↓
Logistic Regression
```

そして、

```python
cross_validate()
```

まで実行してください。

---

# 🌱 今日のまとめ

今日の核心は、

> **前処理もモデルの一部として、再現可能なPipelineにする**

です。

特に覚えておきたいのはこの形。

```text
生データ
 ↓
train/test split
 ↓
ColumnTransformer
 ↓
欠損値処理
 ↓
Encoding / Scaling
 ↓
Model
 ↓
評価
```

そして、

```text
train
 ↓
fit_transform

test
 ↓
transform
```

という原則。

これによって、**testデータの情報を前処理に漏らさない**ようにします。

---

# 🧭 AI工学101の現在地

ここまででscikit-learn編は、

```text
NumPy
 ↓
回帰
 ↓
分類
 ↓
評価指標
 ↓
Pipeline
 ↓
特徴量
 ↓
Cross Validation
 ↓
Grid Search
 ↓
汎化・過学習
 ↓
Decision Tree
 ↓
Random Forest
 ↓
Gradient Boosting
 ↓
評価設計
 ↓
Threshold
 ↓
実データ前処理 ★
```

まで来ました。

ここから先は、**「モデルを動かす」だけでなく「データからモデルまでの一連のシステムを組める」**段階に入っています。

---

# 🔜 第27回

## 特徴量選択と次元削減：情報を残しながら入力を整理する

次回は、

* 不要な特徴量
* 特徴量選択
* `SelectKBest`
* 相関と冗長性
* PCA（主成分分析）
* 次元削減
* 「特徴量を増やす」と「特徴量を減らす」の両方の考え方

を扱います。

ここで初めて、**「モデルに何を食わせるか」を整理して設計する**という特徴量設計のもう一段深い部分に入ります。